# VirtualiZarr → Icechunk in Cloud Storage (Source Coop)

This notebook creates an Icechunk store on Source Coop containing virtual references to Copernicus Marine Service chlorophyll data. The Icechunk store references the original Copernicus data rather than copying it.

## Key Points
- Virtual references point to Copernicus cloudferro S3/HTTPS URLs
- No data duplication - only metadata stored in Icechunk
- Source Coop provides cloud storage for the Icechunk repository
- Users can access the data from anywhere with proper Copernicus credentials

In [ ]:
!pip install -qU icechunk virtualizarr copernicusmarine xarray obstore obspec_utils

In [ ]:
import warnings
import time
from pathlib import Path
import json

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings('ignore', category=UserWarning)

## Get Copernicus file URLs

Use `copernicusmarine` to get a list of files without downloading them.

In [ ]:
# Get file list for July 2024 (example - adjust dates as needed)
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*202407*.nc" \
  --create-file-list copernicus_files_sc.txt

In [ ]:
# Read and convert URLs
with open('copernicus_files_sc.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

s3_urls.sort()

COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"
def s3_to_https(s3_url):
    if s3_url.startswith('s3://'):
        return f"{COPERNICUS_ENDPOINT}/{s3_url[5:]}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
print(f"Found {len(https_urls)} files")
print(f"First: {https_urls[0]}")
print(f"Last: {https_urls[-1]}")

## Set up remote file configuration

Configure how to access the Copernicus cloudferro files.

In [ ]:
# Create object-store handle for the REMOTE Copernicus files
url_prefix = f"{COPERNICUS_ENDPOINT}/"
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})
parser = HDFParser()

# Configure virtual chunk container
# This tells Icechunk where the actual data chunks live
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),
    )
)

print(f"✓ Remote storage configured for: {url_prefix}")

## Set up Source Coop storage

Source Coop provides cloud storage for the Icechunk repository.

**Important**: 
1. Get your Source Coop credentials from the 'View Credentials' link
2. Create a `source-creds.json` file (add to `.gitignore`)
3. Never hard-code credentials in notebooks

In [ ]:
# Read Source Coop credentials from JSON file
with open("source-creds.json") as f:
    source_creds = json.load(f)

# Source Coop bucket information
# Adjust these to match your Source Coop organization and dataset
source_bucket = "us-west-2.opendata.source.coop"
source_prefix = "your-org/copernicus-marine/chlorophyll"  # CHANGE THIS
source_region = "us-west-2"

# Create S3 storage configuration
storage = icechunk.s3_storage(
    bucket=source_bucket,
    prefix=source_prefix,
    region=source_creds["region_name"],
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

print("✓ Source Coop storage configured")

In [ ]:
# Create or open Icechunk repository
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception as e:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a writable session
session = repo.writable_session(branch="main")
print("✓ Ready to write data")

## Write files to Icechunk

Process each file: virtualize and write/append to Icechunk.

In [ ]:
# Process all files
commit_every = 10  # Commit every N files
total_added = 0
start = time.perf_counter()

for i, url in enumerate(https_urls):
    filename = Path(url).name
    
    print(f"[{i+1}/{len(https_urls)}] Processing {filename}...")
    
    # Open file virtually (no data download)
    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=['time', 'lat', 'lon', 'latitude', 'longitude'],
        decode_times=True,
    )
    
    # First file: create, subsequent: append
    if i == 0:
        vds.virtualize.to_icechunk(session.store)
    else:
        vds.virtualize.to_icechunk(session.store, append_dim="time")
    
    total_added += 1
    
    # Commit periodically
    if total_added % commit_every == 0:
        elapsed = time.perf_counter() - start
        snapshot_id = session.commit(f"Add through file {i + 1}")
        print(f"  Committed {total_added} files in {elapsed:.2f}s. Snapshot: {snapshot_id}")
        session = repo.writable_session("main")
        start = time.perf_counter()

# Final commit if needed
if total_added % commit_every != 0:
    snapshot_id = session.commit(f"Final commit: {total_added} files")
    print(f"Final commit: {snapshot_id}")

print(f"\n✓ Successfully added {total_added} files to Icechunk")

## Summary

✓ Created Icechunk repository on Source Coop
✓ Stored virtual references to Copernicus cloudferro data
✓ No data duplication - only metadata in Icechunk

### Next Steps

Users can now access this data from anywhere using:

```python
import icechunk
import xarray as xr

url = "https://data.source.coop/your-org/copernicus-marine/chlorophyll"
storage = icechunk.http_storage(url)

# Provide credentials for virtual chunk access (Copernicus data)
credentials = icechunk.containers_credentials({
    "https://s3.waw3-1.cloudferro.com/": icechunk.http_store()
})

repo = icechunk.Repository.open(
    storage,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, consolidated=False)
```